
This notebook demonstrates basic SQL operations using SQLite in Python. It includes database setup, schema creation, inserts, queries, updates, deletes, and transaction handling.


Install any required packages and import `sqlite3` and `pandas` for SQL execution and result display.

In [1]:
!pip install sqlalchemy
!pip install ipython-sql

print('sqlalchemy and pandas imported successfully')

sqlalchemy and pandas imported successfully


In [4]:
from sqlalchemy import create_engine
engine = create_engine('sqlite:///HospitalManagementDB.sqlite')
connection = engine.connect()


Create a connection to a local SQLite database file and instantiate a cursor object for SQL execution.

In [5]:
# Load the SQL extension for Jupyter
%load_ext sql

In [6]:
# Connect to the SQLite database
%sql sqlite:///HospitalManagementDB.sqlite

In [7]:
%%sql
CREATE TABLE patients (
    patient_id INT PRIMARY KEY,
    age SMALLINT NOT NULL,
    gender CHAR(1) NOT NULL,
    city VARCHAR(50) NOT NULL,
    insurance_provider VARCHAR(50) NOT NULL,
    chronic_flag BIT NOT NULL,
    registration_date DATE NOT NULL
)

 * sqlite:///HospitalManagementDB.sqlite
Done.


[]

In [6]:

%%sql
CREATE TABLE visits (
    visit_id INT PRIMARY KEY,
    patient_id INT NOT NULL,
    visit_date DATE NOT NULL,
    department VARCHAR(50) NOT NULL,
    visit_type VARCHAR(20) NOT NULL,
    length_of_stay_hours DECIMAL(6,2) NOT NULL,
    risk_score VARCHAR(10) NOT NULL,
    doctor_id INT NOT NULL,
    FOREIGN KEY (patient_id) REFERENCES patients(patient_id)
)

 * sqlite:///HospitalManagementDB.sqlite
Done.


[]

In [8]:
%%sql
CREATE TABLE billing (
    bill_id INT PRIMARY KEY,
    visit_id INT NOT NULL,
    billed_amount DECIMAL(12,2) NOT NULL,
    approved_amount DECIMAL(12,2),
    claim_status VARCHAR(20) NOT NULL,
    payment_days INT,
    billing_date DATE NOT NULL,
    FOREIGN KEY (visit_id) REFERENCES visits(visit_id)
)

 * sqlite:///HospitalManagementDB.sqlite
Done.


[]

In [9]:
%%sql
ALTER TABLE billing
ADD CONSTRAINT fk_billing_visit
FOREIGN KEY (visit_id)
REFERENCES visits(visit_id)

ALTER TABLE visits
ADD CONSTRAINT fk_visits_patient
FOREIGN KEY (patient_id)
REFERENCES patients(patient_id)

 * sqlite:///HospitalManagementDB.sqlite
(sqlite3.OperationalError) near "CONSTRAINT": syntax error
[SQL: ALTER TABLE billing
ADD CONSTRAINT fk_billing_visit
FOREIGN KEY (visit_id)
REFERENCES visits(visit_id)

ALTER TABLE visits
ADD CONSTRAINT fk_visits_patient
FOREIGN KEY (patient_id)
REFERENCES patients(patient_id)]
(Background on this error at: https://sqlalche.me/e/20/e3q8)


In [7]:
import pandas as pd
df = pd.read_csv('patients.csv')
df.to_sql('patients', connection, if_exists='append', index=False)

DatabaseError: Execution failed on sql 'INSERT INTO patients (patient_id, age, gender, city, insurance_provider, chronic_flag, registration_date) VALUES (:patient_id, :age, :gender, :city, :insurance_provider, :chronic_flag, :registration_date)': (sqlite3.IntegrityError) UNIQUE constraint failed: patients.patient_id
[SQL: INSERT INTO patients (patient_id, age, gender, city, insurance_provider, chronic_flag, registration_date) VALUES (?, ?, ?, ?, ?, ?, ?)]
[parameters: [(1, 53, 'M', 'Hyderabad', 'SecureLife', 0, '2025-05-14'), (2, 42, 'M', 'Pune', 'HealthPlus', 0, '2025-11-18'), (3, 56, 'F', 'Hyderabad', 'HealthPlus', 0, '2025-05-11'), (4, 72, 'M', 'Chennai', 'HealthPlus', 0, '2025-12-18'), (5, 40, 'F', 'Bangalore', 'CareOne', 1, '2025-01-23'), (6, 40, 'M', 'Pune', 'CareOne', 1, '2025-06-11'), (7, 73, 'M', 'Bangalore', 'SecureLife', 0, '2025-03-08'), (8, 58, 'F', 'Hyderabad', 'SecureLife', 1, '2025-11-25')  ... displaying 10 of 5000 total bound parameter sets ...  (4999, 59, 'M', 'Pune'

In [11]:
df_visits = pd.read_csv('visits.csv')
df_visits.to_sql('visits', connection, if_exists='append', index=False)

25000

In [12]:
df_billing = pd.read_csv('billing.csv')
df_billing.to_sql('billing', connection, if_exists='append', index=False)

25000

In [8]:
# read the data from the database to verify that it was inserted correctly
patients_df = pd.read_sql('SELECT * FROM patients', connection)
print(patients_df)

      patient_id  age gender       city insurance_provider  chronic_flag  \
0              1   53      M  Hyderabad         SecureLife             0   
1              2   42      M       Pune         HealthPlus             0   
2              3   56      F  Hyderabad         HealthPlus             0   
3              4   72      M    Chennai         HealthPlus             0   
4              5   40      F  Bangalore            CareOne             1   
...          ...  ...    ...        ...                ...           ...   
4995        4996   44      F  Hyderabad         SecureLife             1   
4996        4997   57      F  Bangalore            CareOne             1   
4997        4998   90      F    Chennai          MediCareX             0   
4998        4999   59      M       Pune         HealthPlus             1   
4999        5000   29      M    Chennai          MediCareX             0   

     registration_date  
0           2025-05-14  
1           2025-11-18  
2           

## Operational Analysis

In [9]:
# Retrieve the top 10 departments by total visit volume.
top_departments_query = """
    SELECT department,
           COUNT(*) AS total_visits
    FROM visits
    GROUP BY department
    ORDER BY total_visits DESC
    LIMIT 10
"""
top_departments_result = pd.read_sql(top_departments_query, connection)
print(top_departments_result)


    department  total_visits
0      General          4228
1           ER          4220
2    Neurology          4165
3  Orthopedics          4164
4   Cardiology          4159
5          ICU          4064


In [ ]:


# Identify the top 5 departments with the highest average length of stay.
top_departments_los_query = """
    SELECT department,
           AVG(length_of_stay_hours) AS average_length_of_stay
    FROM visits
    GROUP BY department
    ORDER BY average_length_of_stay DESC
    LIMIT 5
"""
top_departments_los_result = pd.read_sql(top_departments_los_query, connection)
print(top_departments_los_result)



In [ ]:

#Find the percentage of High Risk visits per department.
high_risk_percentage_query = """
    SELECT department,
           (SUM(CASE WHEN risk_score = 'High' THEN 1 ELSE 0 END) * 100.0 / COUNT(*)) AS high_risk_percentage
    FROM visits
    GROUP BY department
"""
high_risk_percentage_result = pd.read_sql(high_risk_percentage_query, connection)
print(high_risk_percentage_result)


In [22]:

#Determine the average number of visits per patient by city.
average_visits_per_patient_query = """
    SELECT p.city, AVG(v.visit_count) AS avg_visits_per_patient
    FROM patients p
    JOIN (
        SELECT patient_id, COUNT(*) AS visit_count
        FROM visits
        GROUP BY patient_id
    ) v ON p.patient_id = v.patient_id
    GROUP BY p.city
"""
average_visits_per_patient_result = pd.read_sql(average_visits_per_patient_query, connection)
print(average_visits_per_patient_result)


    department  high_risk_percentage
0   Cardiology             18.994951
1           ER             20.663507
2      General             19.843898
3          ICU             20.792323
4    Neurology             20.312125
5  Orthopedics             20.220941


In [23]:


#Identify doctors handling the highest number of High Risk visits.
high_risk_doctors_query = """
    SELECT doctor_id, COUNT(*) AS high_risk_visit_count
    FROM visits
    WHERE risk_score = 'High'
    GROUP BY doctor_id
    ORDER BY high_risk_visit_count DESC
    LIMIT 5
"""
high_risk_doctors_result = pd.read_sql(high_risk_doctors_query, connection)
print(high_risk_doctors_result)

# Retrieve the top 10 insurance providers by total billed amount


        city  avg_visits_per_patient
0  Bangalore                5.023895
1    Chennai                5.018939
2      Delhi                4.954162
3  Hyderabad                5.057870
4     Mumbai                5.020706
5       Pune                5.122573


## Financial Analysis

In [ ]:
# Retrieve the top 10 insurance providers by total billed amount
 